# Azerbaijani (aze) — Full NLP Pipeline with Custom Stanza Models

Azerbaijani uses the Latin script in Azerbaijan (official since 1991) and the Cyrillic script in Russia. TurkicNLP provides Apertium FST morphology (Stable), custom-trained Stanza neural models for POS tagging, lemmatisation, and dependency parsing, bidirectional Cyrillic↔Latin transliteration, and NLLB-200 embeddings/translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('aze')

## 2. Cyrillic ↔ Latin Transliteration (1991 standard)

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("AZERBAIJANI COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

cyrl = "Мən məktəbə gedirəm."
print(f"Original (Cyrillic, used in Russia): {cyrl}")
print()

# Direction 1: Cyrillic → Turkic Common Alphabet (Latin, 1991 official)
print("1. Cyrillic → Turkic Common Alphabet (Latin, 1991 official):")
try:
    t1 = Transliterator("aze", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(cyrl)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Cyrillic (reverse)
print("2. Turkic Common (Latin) → Cyrillic:")
try:
    t2 = Transliterator("aze", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    back_to_cyrl = t2.transliterate(common if 'common' in locals() else "Mən məktəbə gedirəm.")
    print(f"   {back_to_cyrl}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Cyrillic → Latin
print("3. Cyrillic → Latin (explicit):")
try:
    t3 = Transliterator("aze", source=Script.CYRILLIC, target=Script.LATIN)
    latin = t3.transliterate(cyrl)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Cyrillic
print("4. Latin → Cyrillic:")
try:
    t4 = Transliterator("aze", source=Script.LATIN, target=Script.CYRILLIC)
    back_to_cyrl_explicit = t4.transliterate(latin if 'latin' in locals() else "Mən məktəbə gedirəm.")
    print(f"   {back_to_cyrl_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Azerbaijani Scripts:")
print("  • Latin (official in Azerbaijan since 1991, COMMON_TURKIC standard)")
print("  • Cyrillic (used in Russia, minority communities)")
print("=" * 70)

## 3. Morphological Analysis (Apertium FST — Stable)

In [ ]:
nlp = Pipeline(
    "aze",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Mən məktəbə gedirəm.")
for w in doc.words:
    print(f"{w.text:<20} lemma={w.lemma:<15} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (Custom Stanza)

TurkicNLP includes custom-trained Stanza models for Azerbaijani, providing POS tagging, lemmatisation, and dependency parsing.

In [ ]:
nlp_parse = Pipeline(
    "aze",
    processors=["tokenize", "pos", "lemma", "depparse"],
)

doc = nlp_parse("Bakı Azərbaycanın paytaxtıdır.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Head':<5} {'Deprel'}")
print("-" * 60)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.head!s:<5} {w.deprel}")

## 5. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "aze",
    processors=["tokenize", "morph", "pos", "lemma", "depparse"],
    morph_backend="apertium",
)
doc = nlp_full("Azərbaycan Cənubi Qafqazda yerləşən bir dövlətdir.")
print(doc.to_conllu())

## 6. Translation

In [ ]:
turkicnlp.download("aze", processors=["translate"])
trans = Pipeline("aze", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Azərbaycan Cənubi Qafqazda yerləşən bir dövlətdir.")
print("EN:", doc.translation)